In [3]:
from groq import Groq

import json
from datetime import datetime


In [ ]:
client = Groq(api_key=key)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one sentence"}
    ]
)


print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [5]:


client = Groq(api_key=key)

# Step 1: The actual Python function
def get_current_date():
    return datetime.now().strftime("%A, %B %d, %Y")

# Step 2: Tell Groq this function exists (just a description, not the actual function)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Returns today's date. Call this when the user asks anything about today's date or current day.",
            "parameters": {
                "type": "object",
                "properties": {},  # No inputs needed for this function
                "required": []
            }
        }
    }
]

# Step 3: Send message + tools to Groq
messages = [
    {"role": "user", "content": "What day is it today?"}
]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools
)

# Step 4: Check if Groq wants to call a function
response_message = response.choices[0].message

if response_message.tool_calls:
    print("Groq wants to call a function!")
    tool_call = response_message.tool_calls[0]
    print(f"Function name: {tool_call.function.name}")
    
    # Step 5: Actually run the function
    result = get_current_date()
    print(f"Function returned: {result}")
    
    # Step 6: Send the result back to Groq
    messages.append(response_message)  # Add Groq's "I want to call X" message
    messages.append({                   # Add the function result
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })
    
    # Step 7: Get Groq's final response now that it has the date
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools
    )
    print(f"\nGroq's final answer: {final_response.choices[0].message.content}")

else:
    # Groq answered directly without needing the function
    print(f"Groq answered directly: {response_message.content}")

Groq wants to call a function!
Function name: get_current_date
Function returned: Saturday, August 15, 2026

Groq's final answer: It's Saturday, August 15, 2026.
